<a href="https://colab.research.google.com/github/baubyte/CienciaDeDatos/blob/main/Semana_4/Ejercicios/Taller_centros_de_salud_v3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Taller: Integración y análisis de datos de un conglomerado de centros de salud

## Contexto

Un conglomerado de centros de salud distribuidos en distintas provincias de Argentina necesita consolidar la información de sus consultas médicas para tomar decisiones de gestión. Los datos provienen de tres fuentes distintas:

- **consultas.csv**: registro de consultas médicas realizadas durante el año.
- **prestaciones.csv**: catálogo de prestaciones médicas con sus categorías y costos de referencia.
- **sucursales.txt**: listado de centros de salud con su ubicación geográfica.

El objetivo del taller es integrar estas fuentes, limpiar los datos y responder una pregunta de análisis asignada a cada grupo.

---

## Parte 1: Lectura y exploración de los datos

### 1.1 Importar las bibliotecas necesarias

In [1]:
# Importar la biblioteca necesaria
import pandas as pd
from google.colab import userdata, drive, files

#Montar Drive
drive.mount("/content/drive")



Mounted at /content/drive


In [2]:
url_dataset:str ="https://raw.githubusercontent.com/baubyte/CienciaDeDatos/refs/heads/main/Semana_4/Ejercicios/practica_centros.zip"
dir_dataset:str = 'drive/MyDrive/kaggle/datasets'
folder_dataset:str = 'practica-centros'
#Descargamos el dataset
!wget -nc {url_dataset} -P {dir_dataset}
# Descomprimir el dataset para acceder a los archivos CSV y TXT
!unzip -o {dir_dataset}"/practica_centros.zip" -d {dir_dataset}
#Elimnamos el zip
!rm {dir_dataset}"/practica_centros.zip"


--2026-04-11 20:42:55--  https://raw.githubusercontent.com/baubyte/CienciaDeDatos/refs/heads/main/Semana_4/Ejercicios/practica_centros.zip
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.110.133, 185.199.111.133, 185.199.108.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.110.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 14530 (14K) [application/zip]
Saving to: ‘drive/MyDrive/kaggle/datasets/practica_centros.zip’

practica_centros.zi 100%[===================>]  14.19K  --.-KB/s    in 0.003s  

2026-04-11 20:42:55 (5.28 MB/s) - ‘drive/MyDrive/kaggle/datasets/practica_centros.zip’ saved [14530/14530]

Archive:  drive/MyDrive/kaggle/datasets/practica_centros.zip
  inflating: drive/MyDrive/kaggle/datasets/practica-centros/consultas.csv  
  inflating: drive/MyDrive/kaggle/datasets/practica-centros/prestaciones.csv  
  inflating: drive/MyDrive/kaggle/datasets/practica-centros/sucursales.txt  
  in

### 1.2 Leer el archivo de consultas

Leer el archivo `consultas.csv` en un DataFrame y explorar su contenido con `.shape`, `.dtypes`, `.head()` e `.info()`.

**Preguntas guía:**
- ¿Cuántas filas y columnas tiene el dataset?
- ¿Qué tipo de dato asignó Pandas a cada columna?
- ¿Hay valores faltantes? ¿En qué columnas?

In [3]:
# Leer el archivo CSV de consultas
df_consultas = pd.read_csv(f"{dir_dataset}/{folder_dataset}/consultas.csv")

In [4]:
# Explorar el DataFrame: shape, dtypes, head, info

print(f"Cantidad de filas: {df_consultas.shape[0]}")
print(f"Cantidad de columnas: {df_consultas.shape[1]}")


Cantidad de filas: 650
Cantidad de columnas: 8


In [45]:
print("Tipos de Datos Columnas: ")
#print(consultas.dtypes)
df_consultas.info()
print("Columnas con datos faltantes")
df_consultas.isnull().sum()

Tipos de Datos Columnas: 
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 650 entries, 0 to 649
Data columns (total 8 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   id_consulta    650 non-null    object
 1   fecha          650 non-null    object
 2   id_centro      650 non-null    object
 3   id_prestacion  650 non-null    object
 4   costo          650 non-null    int64 
 5   obra_social    650 non-null    object
 6   edad_paciente  650 non-null    int64 
 7   diagnostico    567 non-null    object
dtypes: int64(2), object(6)
memory usage: 40.8+ KB
Columnas con datos faltantes


,0
id_consulta,0
fecha,0
id_centro,0
id_prestacion,0
costo,0
obra_social,0
edad_paciente,0
diagnostico,83


In [6]:
print("Primeras filas: ")
df_consultas.head()

Primeras filas: 


,id_consulta,fecha,id_centro,id_prestacion,costo,obra_social,edad_paciente,diagnostico
0,CON00001,22/01/2025,CS006,P005,12467,Swiss Medical,12,Contractura muscular
1,CON00002,28/12/2024,CS001,P002,8423,OSDE,54,Contractura muscular
2,CON00003,21/06/2024,CS003,P013,$8886,Galeno,36,Alergia estacional
3,CON00004,19/05/2024,CS003,P002,8076,Particular,59,Hipertensión arterial
4,CON00005,30/11/2024,CS004,P002,$9344,Particular,6,Diabetes tipo 2


In [7]:
# Filtrar y mostrar solo las columnas con valores nulos
print("Columnas con datos faltantes y sus cantidades:")
null_counts = df_consultas.isnull().sum()
only_nulls = null_counts[null_counts > 0]
display(only_nulls)

Columnas con datos faltantes y sus cantidades:


,0
diagnostico,83


### 1.3 Leer el archivo de prestaciones

Leer el archivo `prestaciones.csv` y explorar su contenido.

**Preguntas guía:**
- ¿Qué columnas contiene?
- ¿Qué columna podría usarse para vincularlo con el archivo de consultas?

In [8]:
# Leer el archivo CSV de prestaciones
df_prestaciones = pd.read_csv(f"{dir_dataset}/{folder_dataset}/prestaciones.csv")


In [10]:
print("Columnas Dataset Prestaciones", df_prestaciones.columns.to_list())

Columnas Dataset Prestaciones ['id_prestacion', 'nombre_prestacion', 'categoria', 'costo_base']


In [22]:
print("Se puede vincular con consultas por la columna id_prestacion")

Se puede vincular con consultas por la columna id_prestacion


In [21]:
print(f"Cantidad de filas: {df_prestaciones.shape[0]}")
print(f"Cantidad de columnas: {df_prestaciones.shape[1]}")

Cantidad de filas: 15
Cantidad de columnas: 4


In [12]:
print("Primeras filas Dataset Prestaciones")
display(df_prestaciones.head())

Primeras filas Dataset Prestaciones


,id_prestacion,nombre_prestacion,categoria,costo_base
0,P001,Consulta clínica general,Clínica,8500
1,P002,Consulta pediátrica,Pediatría,9200
2,P003,Control cardiológico,Cardiología,15000
3,P004,Consulta dermatológica,Dermatología,11000
4,P005,Consulta traumatológica,Traumatología,13500


In [47]:
# Explorar el DataFrame de prestaciones
print("Tipos de Datos Columnas: ")
df_prestaciones.info()
print("Columnas con datos faltantes")
df_prestaciones.isnull().sum()

Tipos de Datos Columnas: 
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15 entries, 0 to 14
Data columns (total 4 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   id_prestacion      15 non-null     object
 1   nombre_prestacion  15 non-null     object
 2   categoria          15 non-null     object
 3   costo_base         15 non-null     int64 
dtypes: int64(1), object(3)
memory usage: 612.0+ bytes
Columnas con datos faltantes


,0
id_prestacion,0
nombre_prestacion,0
categoria,0
costo_base,0


### 1.4 Leer el archivo de centros de salud

Leer el archivo `sucursales.txt`. Tener en cuenta que está delimitado por tabulaciones.

**Preguntas guía:**
- ¿Qué separador se debe indicar para leer correctamente este archivo?
- ¿Los nombres de los centros están limpios o tienen algún problema?

In [17]:
# Leer el archivo TXT de centros de salud
# Sugerencia: investigar el parámetro sep de pd.read_csv()

print('Lectura Dateset sucursales.txt, usando separador \"tab\"')
df_sucursales = pd.read_csv(f"{dir_dataset}/{folder_dataset}/sucursales.txt",sep='\t')


Lectura Dateset sucursales.txt, usando separador "tab"


In [36]:
print("Primeras filas Dataset Sucursales")
print(df_sucursales)

Primeras filas Dataset Sucursales
  id_centro                   nombre_centro                        localidad  \
0     CS001       Centro de Salud Rivadavia  Ciudad Autónoma de Buenos Aires   
1     CS002           Centro Médico del Sur                         La Plata   
2     CS003              Clínica San Martín                          Córdoba   
3     CS004        Centro de Salud Belgrano                          Rosario   
4     CS005               Policlínico Norte                          Mendoza   
5     CS006  Centro Integral de Salud Oeste  Ciudad Autónoma de Buenos Aires   

      provincia       telefono  
0  Buenos Aires  011-4555-1234  
1  Buenos Aires  0221-422-5678  
2       Córdoba  0351-489-0012  
3      Santa Fe  0341-456-3344  
4       Mendoza  0261-411-7788  
5  Buenos Aires  011-4600-9900  


In [28]:
print("El nombre de los centros tiene espacios de mas")

El nombre de los centros tiene espacios de mas


In [24]:
print(f"Cantidad de filas: {df_sucursales.shape[0]}")
print(f"Cantidad de columnas: {df_sucursales.shape[1]}")

Cantidad de filas: 6
Cantidad de columnas: 5


In [48]:
print("Tipos de Datos Columnas: ")
df_sucursales.info()
print("Columnas con datos faltantes")
df_sucursales.isnull().sum()

Tipos de Datos Columnas: 
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6 entries, 0 to 5
Data columns (total 5 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   id_centro      6 non-null      object
 1   nombre_centro  6 non-null      object
 2   localidad      6 non-null      object
 3   provincia      6 non-null      object
 4   telefono       6 non-null      object
dtypes: object(5)
memory usage: 372.0+ bytes
Columnas con datos faltantes


,0
id_centro,0
nombre_centro,0
localidad,0
provincia,0
telefono,0


---

## Parte 2: Limpieza de datos

### 2.1 Limpiar la columna de costos

La columna `costo` del archivo de consultas no es numérica: contiene símbolos de pesos (`$`), espacios y puntos como separadores de miles.

**Tareas:**
1. Eliminar los caracteres no numéricos de la columna `costo`.
2. Convertir la columna a tipo numérico (`int` o `float`).
3. Verificar que la conversión fue exitosa.

In [31]:
# Limpiar y convertir la columna de costos
# Sugerencia: investigar el método .str.replace() de Pandas
# Pista: hay que eliminar '$', '.' y espacios antes de convertir

df_consultas['costo'] = df_consultas['costo'].str.replace(
    '$', '').str.replace('.', '').str.replace(' ', '')
df_consultas['costo'] = pd.to_numeric(df_consultas['costo'])

df_consultas.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 650 entries, 0 to 649
Data columns (total 8 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   id_consulta    650 non-null    object
 1   fecha          650 non-null    object
 2   id_centro      650 non-null    object
 3   id_prestacion  650 non-null    object
 4   costo          650 non-null    int64 
 5   obra_social    650 non-null    object
 6   edad_paciente  650 non-null    int64 
 7   diagnostico    567 non-null    object
dtypes: int64(2), object(6)
memory usage: 40.8+ KB


### 2.2 Limpiar los nombres de los centros de salud

Algunos nombres de centros tienen espacios dobles o espacios al inicio/final.

**Tareas:**
1. Eliminar espacios extra en la columna `nombre_centro`.
2. Verificar el resultado comparando antes y después.

In [33]:
# Limpiar los nombres de centros de salud
# Sugerencia: investigar .str.strip() y .str.replace() con expresiones regulares
df_sucursales['nombre_centro'] = df_sucursales['nombre_centro'].str.strip(
).str.replace(' +', ' ', regex=True)

### 2.3 Diagnóstico de los datos limpios

Redactar un breve resumen que describa:
- Qué problemas se encontraron en cada archivo.
- Cómo se resolvió cada uno.
- El estado final de cada DataFrame (cantidad de filas, columnas y tipos de datos).

**1. Archivo `consultas.csv` (`df_consultas`)**
- **Problemas:** La columna `costo` venía como tipo texto (`object`) porque incluía el signo peso (`$`), puntos y espacios. Adicionalmente, se detectaron 83 valores faltantes en la columna `diagnostico`.
- **Resolución:** Para limpiar `costo`, se eliminaron los caracteres no numéricos utilizando el método `.str.replace()` y luego la columna fue convertida a un formato numérico usando `pd.to_numeric()`.
- **Estado final:** DataFrame de 650 filas y 8 columnas. Los tipos de datos quedaron como `int64` para las columnas `edad_paciente` y `costo`, y `object` para las 6 columnas restantes.

**2. Archivo `sucursales.txt` (`sucursales`)**
- **Problemas:** El primer inconveniente fue que el archivo estaba delimitado por tabulaciones en vez de comas. Además, la columna `nombre_centro` tenía problemas de formato, conteniendo espacios sobrantes al inicio/final y espacios dobles entre las palabras.
- **Resolución:** Se resolvió la lectura indicando el parámetro `sep='\t'`. Los espacios en `nombre_centro` se limpiaron encadenando `.str.strip()` (para los extremos) y `.str.replace(' +', ' ')` (para unificar múltiples espacios internos en uno solo).
- **Estado final:** DataFrame con 6 filas y 5 columnas. Todas sus columnas son de tipo de dato `object` (texto).

**3. Archivo `prestaciones.csv` (`df_prestaciones`)**
- **Problemas:** No se encontraron valores nulos ni problemas de formato en sus columnas durante la inspección.
- **Resolución:** No hizo falta aplicarle etapas de limpieza.
- **Estado final:** DataFrame con 15 filas y 4 columnas. Sus tipos de datos son `int64` para la columna `costo_base` y `object` para las otras 3 columnas.

---

## Parte 3: Integración de los datos

### 3.1 Unir consultas con prestaciones

Realizar un join entre el DataFrame de consultas y el de prestaciones.

**Preguntas guía:**
- ¿Qué columna tienen en común ambos DataFrames?
- ¿Qué tipo de join conviene usar: `inner`, `left` o `right`? ¿Por qué?
- ¿Se pierden registros después del join? ¿Qué significaría si eso ocurriera?

In [58]:
# Unir consultas con prestaciones
# Sugerencia: usar pd.merge() con los parámetros on= y how=
df_consultas_prestaciones = pd.merge(df_consultas, df_prestaciones, on='id_prestacion')
display(df_consultas_prestaciones)

,id_consulta,fecha,id_centro,id_prestacion,costo,obra_social,edad_paciente,diagnostico,nombre_prestacion,categoria,costo_base
0,CON00001,22/01/2025,CS006,P005,12467,Swiss Medical,12,Contractura muscular,Consulta traumatológica,Traumatología,13500
1,CON00002,28/12/2024,CS001,P002,8423,OSDE,54,Contractura muscular,Consulta pediátrica,Pediatría,9200
2,CON00003,21/06/2024,CS003,P013,8886,Galeno,36,Alergia estacional,Consulta nutricional,Nutrición,8000
3,CON00004,19/05/2024,CS003,P002,8076,Particular,59,Hipertensión arterial,Consulta pediátrica,Pediatría,9200
4,CON00005,30/11/2024,CS004,P002,9344,Particular,6,Diabetes tipo 2,Consulta pediátrica,Pediatría,9200
...,...,...,...,...,...,...,...,...,...,...,...
645,CON00646,09/07/2024,CS004,P008,9738,Medifé,41,Cefalea tensional,Consulta psicológica,Salud Mental,9000
646,CON00647,06/12/2024,CS001,P014,8704,Swiss Medical,29,Gastritis,Kinesiología,Rehabilitación,10000
647,CON00648,24/07/2024,CS001,P001,8804,Medifé,82,NaN,Consulta clínica general,Clínica,8500
648,CON00649,2024-05-02,CS001,P002,10184,Galeno,90,NaN,Consulta pediátrica,Pediatría,9200


**Respuestas:**

- **¿Qué columna tienen en común ambos DataFrames?**
La columna en común, y que actúa como clave primaria en prestaciones y clave foránea en consultas, es `id_prestacion`.

- **¿Qué tipo de join conviene usar: `inner`, `left` o `right`? ¿Por qué?**
Para este tipo de negocio, es ideal utilizar un **`inner join`** (por defecto en Pandas). Porque es se deberia mantener la consistencia a nivel lógico, no debería existir ninguna consulta médica registrada que no esté vinculada a una prestación válida (registrada en prestaciones).

- **¿Se pierden registros después del join? ¿Qué significaría si eso ocurriera?**
En este caso **no se perdieron registros** porque el DataFrame resultado del merge sigue teniendo 650 filas  esto comprueba que los ids de prestaciones cargados en las consultas son congruentes.
Si se hubieran perdido registros al hacer un *inner join*, significaría que existen problemas de integridad referencial: habría consultas médicas con un código asignado inexistente.


### 3.2 Unir con centros de salud

Agregar al DataFrame consolidado la información de los centros de salud (nombre del centro, localidad, provincia).

**Tareas:**
1. Realizar el segundo join.
2. Verificar la cantidad de filas resultante.
3. Explorar el DataFrame consolidado final.

In [74]:
# Unir con centros de salud
df_consultas_prestaciones_centros = pd.merge(
    df_consultas_prestaciones, df_sucursales[['nombre_centro', 'localidad', 'provincia', 'id_centro']], on='id_centro')

In [75]:
# Verificar el DataFrame consolidado
print(f"Cantidad de filas originales: {df_consultas_prestaciones.shape[0]}")
print( f"Cantidad de filas finales: {df_consultas_prestaciones_centros.shape[0]}")

display(df_consultas_prestaciones_centros)

Cantidad de filas originales: 650
Cantidad de filas finales: 650


,id_consulta,fecha,id_centro,id_prestacion,costo,obra_social,edad_paciente,diagnostico,nombre_prestacion,categoria,costo_base,nombre_centro,localidad,provincia
0,CON00001,22/01/2025,CS006,P005,12467,Swiss Medical,12,Contractura muscular,Consulta traumatológica,Traumatología,13500,Centro Integral de Salud Oeste,Ciudad Autónoma de Buenos Aires,Buenos Aires
1,CON00002,28/12/2024,CS001,P002,8423,OSDE,54,Contractura muscular,Consulta pediátrica,Pediatría,9200,Centro de Salud Rivadavia,Ciudad Autónoma de Buenos Aires,Buenos Aires
2,CON00003,21/06/2024,CS003,P013,8886,Galeno,36,Alergia estacional,Consulta nutricional,Nutrición,8000,Clínica San Martín,Córdoba,Córdoba
3,CON00004,19/05/2024,CS003,P002,8076,Particular,59,Hipertensión arterial,Consulta pediátrica,Pediatría,9200,Clínica San Martín,Córdoba,Córdoba
4,CON00005,30/11/2024,CS004,P002,9344,Particular,6,Diabetes tipo 2,Consulta pediátrica,Pediatría,9200,Centro de Salud Belgrano,Rosario,Santa Fe
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
645,CON00646,09/07/2024,CS004,P008,9738,Medifé,41,Cefalea tensional,Consulta psicológica,Salud Mental,9000,Centro de Salud Belgrano,Rosario,Santa Fe
646,CON00647,06/12/2024,CS001,P014,8704,Swiss Medical,29,Gastritis,Kinesiología,Rehabilitación,10000,Centro de Salud Rivadavia,Ciudad Autónoma de Buenos Aires,Buenos Aires
647,CON00648,24/07/2024,CS001,P001,8804,Medifé,82,NaN,Consulta clínica general,Clínica,8500,Centro de Salud Rivadavia,Ciudad Autónoma de Buenos Aires,Buenos Aires
648,CON00649,2024-05-02,CS001,P002,10184,Galeno,90,NaN,Consulta pediátrica,Pediatría,9200,Centro de Salud Rivadavia,Ciudad Autónoma de Buenos Aires,Buenos Aires



### 4.1 Pregunta de análisis

Cada grupo debe responder **una** de las siguientes preguntas:

1- ¿Cuál es la categoría de prestación más frecuente en cada centro de salud?

2- ¿Cuál es el costo promedio de consulta por obra social?

3- ¿Qué obra social concentra el mayor gasto total?

4-¿En qué provincia se realizaron más consultas?

---

### 4.2 Resolución

Utilizar `groupby()` y funciones de agregación (`.sum()`, `.mean()`, `.count()`, etc.) para responder la pregunta asignada.

In [94]:
# Agrupamos por obra social y calculamos el promedio del costo
promedios_obra_sociales = df_consultas_prestaciones_centros.groupby('obra_social')['costo'].mean()

# Lo convertimos a un DataFrame para visualizarlo más limpio, lo redondeamos y ordenamos
promedios_obra_sociales = promedios_obra_sociales.reset_index(
    name='costo_promedio').round(2)
promedios_obra_sociales = promedios_obra_sociales.sort_values(
    by='costo_promedio', ascending=False)

display(promedios_obra_sociales)

,obra_social,costo_promedio
6,SanCor Salud,11483.37
1,IOMA,11296.66
3,OSDE,11096.87
7,Swiss Medical,11070.40
5,Particular,10854.63
4,PAMI,10797.01
0,Galeno,10761.11
2,Medifé,10601.37


In [105]:
# Agrupamos por obra social y sumamos los costos generados
obras_sociales_costo_total = df_consultas_prestaciones_centros.groupby('obra_social')[
    'costo'].sum()
# Lo convertimos a un DataFrame y lo ordenamos de mayor a menor gasto
obras_sociales_costo_total = obras_sociales_costo_total.reset_index(
    name='gasto_total_acumulado')
obras_sociales_costo_total = obras_sociales_costo_total.sort_values(
    by='gasto_total_acumulado', ascending=False)

display(obras_sociales_costo_total)

,obra_social,gasto_total_acumulado
7,Swiss Medical,1062758
5,Particular,1020335
6,SanCor Salud,930153
0,Galeno,903933
1,IOMA,903733
3,OSDE,843362
4,PAMI,798979
2,Medifé,689089


In [108]:
# Agrupamos por provincia y contamos los registros
provincia_mayor_consultas = df_consultas_prestaciones_centros.groupby(
    'provincia').size()
# Convertimos a un DataFrame y ordenamos de mayor a menor cantidad
provincia_mayor_consultas = provincia_mayor_consultas.reset_index(name='cantidad_de_consultas')
provincia_mayor_consultas = provincia_mayor_consultas.sort_values(
    by='cantidad_de_consultas', ascending=False)

display(provincia_mayor_consultas)

,provincia,cantidad_de_consultas
0,Buenos Aires,326
3,Santa Fe,136
1,Córdoba,97
2,Mendoza,91


### 4.3 Interpretación

Escribir una breve interpretación del resultado obtenido (3 a 5 oraciones). ¿Qué le diría este resultado al equipo de gestión de los centros de salud?

*Escribir la interpretación aquí.*